# 05 - Bias Detection

Detect gendered language and elite-university preferences in job descriptions using heuristic analysis.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

In [ ]:
# Define bias term dictionaries
MASCULINE_TERMS = {
    "aggressive", "dominant", "competitive", "assertive", "ninja",
    "rockstar", "guru", "alpha", "maverick", "warrior", "hacker",
    "crush", "kill", "destroy", "execute", "heavy lifting"
}

FEMININE_TERMS = {
    "collaborative", "supportive", "nurturing", "compassionate",
    "interpersonal", "understanding", "loyal", "honest"
}

ELITE_UNIVERSITIES = {
    "harvard", "mit", "stanford", "yale", "princeton",
    "cambridge", "oxford", "caltech", "columbia", "cornell",
    "dartmouth", "brown", "penn", "ivy league"
}

print(f"Masculine terms: {len(MASCULINE_TERMS)}")
print(f"Feminine terms: {len(FEMININE_TERMS)}")
print(f"Elite universities: {len(ELITE_UNIVERSITIES)}")

In [ ]:
# Load bias samples dataset
bias_df = pd.read_csv('../datasets/bias_samples.csv')
print(f"Bias samples: {len(bias_df)}")
print(f"Label distribution: {bias_df['label'].value_counts().to_dict()}")

In [ ]:
# Bias detection functions
def detect_gender_bias(text):
    text_lower = text.lower()
    masc_found = [t for t in MASCULINE_TERMS if t in text_lower]
    fem_found = [t for t in FEMININE_TERMS if t in text_lower]
    return masc_found, fem_found

def detect_university_bias(text):
    text_lower = text.lower()
    found = [u for u in ELITE_UNIVERSITIES if u in text_lower]
    return found

def calculate_fairness_score(masc_count, fem_count, uni_count):
    score = 100.0
    if masc_count > fem_count:
        score -= 15
    if fem_count > 0 and masc_count == 0:
        score -= 5
    score -= uni_count * 10
    score -= min(20, (masc_count + fem_count) * 3)
    return max(0.0, score)

# Test on sample
sample = "Looking for a rockstar ninja who can crush it. Harvard preferred."
masc, fem = detect_gender_bias(sample)
uni = detect_university_bias(sample)
fairness = calculate_fairness_score(len(masc), len(fem), len(uni))

print(f"Sample: {sample}")
print(f"Masculine: {masc}")
print(f"Feminine: {fem}")
print(f"University: {uni}")
print(f"Fairness: {fairness:.1f}%")

In [ ]:
# Apply to full bias dataset
bias_df['masc_terms'] = bias_df['text'].apply(lambda x: detect_gender_bias(x)[0])
bias_df['fem_terms'] = bias_df['text'].apply(lambda x: detect_gender_bias(x)[1])
bias_df['uni_terms'] = bias_df['text'].apply(detect_university_bias)
bias_df['masc_count'] = bias_df['masc_terms'].apply(len)
bias_df['fem_count'] = bias_df['fem_terms'].apply(len)
bias_df['uni_count'] = bias_df['uni_terms'].apply(len)
bias_df['fairness'] = bias_df.apply(
    lambda r: calculate_fairness_score(r['masc_count'], r['fem_count'], r['uni_count']), axis=1
)

print(f"Average fairness score: {bias_df['fairness'].mean():.1f}%")
print(f"Biased samples detected: {(bias_df['label'] == 1).sum()} / {len(bias_df)}")

In [ ]:
# Visualize bias term frequency
all_masc = [t for terms in bias_df['masc_terms'] for t in terms]
masc_counts = Counter(all_masc).most_common(10)

if masc_counts:
    terms, counts = zip(*masc_counts)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=list(counts), y=list(terms), palette='Reds_r')
    plt.title('Most Common Masculine-Coded Terms')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No masculine terms found in dataset")

In [ ]:
# Fairness score distribution
plt.figure(figsize=(8, 5))
sns.histplot(bias_df['fairness'], bins=20, kde=True, color='blue')
plt.axvline(x=80, color='red', linestyle='--', label='Threshold (80%)')
plt.title('Fairness Score Distribution')
plt.xlabel('Fairness Score (%)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

below_threshold = (bias_df['fairness'] < 80).sum()
print(f"Samples below fairness threshold (80%): {below_threshold} / {len(bias_df)}")

In [ ]:
# Pie chart of bias types
gender_biased = (bias_df['masc_count'] > bias_df['fem_count']).sum()
uni_biased = (bias_df['uni_count'] > 0).sum()
clean = len(bias_df) - gender_biased - uni_biased + (bias_df[(bias_df['masc_count'] > bias_df['fem_count']) & (bias_df['uni_count'] > 0)]).shape[0]

labels = ['Gender Bias', 'University Bias', 'Clean']
sizes = [gender_biased, uni_biased, max(0, clean)]
colors = ['#ef4444', '#f97316', '#22c55e']

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Bias Type Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Test on job descriptions
jobs = pd.read_csv('../datasets/jobs.csv')

print("Bias analysis on job descriptions:")
print("=" * 50)
for _, job in jobs.head(10).iterrows():
    masc, fem = detect_gender_bias(job['description'])
    uni = detect_university_bias(job['description'])
    fairness = calculate_fairness_score(len(masc), len(fem), len(uni))
    
    flagged = []
    if masc: flagged.append(f"masculine: {masc}")
    if fem: flagged.append(f"feminine: {fem}")
    if uni: flagged.append(f"university: {uni}")
    
    status = "⚠️" if fairness < 80 else "✅"
    print(f"{status} {job['title'][:30]:30s} | Fairness: {fairness:.0f}% | {' | '.join(flagged) if flagged else 'Clean'}")